In [ ]:
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")
else:
    print("Not running in Colab. Skipping drive mount.")

In [ ]:
!pip install /content/drive/MyDrive/eq_5d/scispacy/en_core_sci_md-0.5.4.tar.gz

In [ ]:
import csv
import os
import re
import random
from typing import List, Optional, Union, Dict, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim, amp
from torch.utils.data import DataLoader, Dataset, SequentialSampler, RandomSampler

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    get_scheduler
)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
MODEL_MAP = {
    "bert": "bert-base-uncased",
    "scibert": "allenai/scibert_scivocab_uncased",
    "biobert": "dmis-lab/biobert-base-cased-v1.2",
    "pubmedbert": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
    "biolinkbert": "michiyasunaga/BioLinkBERT-base"
}
selected_model = "biobert"
BASE_MODEL = MODEL_MAP[selected_model]

In [ ]:
SAVE_DIR = os.environ.get("SAVE_DIR", "/content/drive/MyDrive/Zhi_folders/pubmed_informatics_confernce_paper_plms/")
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Stage 1: SimCSE contrastive pretraining 
PRETRAIN_BATCH_SIZE = 128
PRETRAIN_EPOCHS = 5
PRETRAIN_LR = 3e-5
PRETRAIN_TEMP = 0.05

#  Supervised fine-tuning (shared by all configs) 
FINETUNE_BATCH_SIZE = 128
FINETUNE_EPOCHS = 20
FINETUNE_LRS = [3e-5, 5e-6]
FINETUNE_PATIENCE = 5
FOCAL_GAMMA = 2.0

#  Config 2: MLM domain-adaptive pretraining (DAPT) 
DAPT_BATCH_SIZE = 64
DAPT_EPOCHS = 3
DAPT_LR = 5e-5
DAPT_MLM_PROBABILITY = 0.15

#  Config 3: soft distillation on unlabeled data 
SOFT_DISTILL_EPOCHS = 6
SOFT_DISTILL_LR = 2e-6
SOFT_DISTILL_PATIENCE = 3
SOFT_DISTILL_WEIGHT = 0.5

#  Config 4: Supervised Contrastive Learning (on top of Config 3)
SUPCON_EPOCHS = 6
SUPCON_LR = 2e-6
SUPCON_PATIENCE = 3
SUPCON_WEIGHT = 0.05
SUPCON_TEMPERATURE = 0.1
SUPCON_PROJECTION_DIM = 128
SUPCON_WARMUP_FRACTION = 0.15

#  Evaluation 
BOOTSTRAP_N = 1000
BOOTSTRAP_CI = 0.90

MAX_LENGTH = 256


In [ ]:
def clean_text_for_plm(text: Union[str, None]) -> str:
    if text is None:
        return ""
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[\u0000-\u001F\u007F]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()


In [ ]:
def infer_document_ids(df: pd.DataFrame) -> Optional[pd.Series]:
    candidate_id_cols = ["PMID", "pmid", "abstract_id", "ABSTRACT_ID", "doc_id", "DOC_ID", "id", "ID"]
    for c in candidate_id_cols:
        if c in df.columns:
            print(f"  Using existing '{c}' column as document id.")
            return df[c]

    candidate_pos_cols = ["line_number", "LINE_NUMBER", "sent_id", "SENT_ID",
                           "position", "POSITION", "sentence_id", "SENTENCE_ID"]
    for c in candidate_pos_cols:
        if c in df.columns:
            pos = pd.to_numeric(df[c], errors="coerce")
            if pos.isna().any():
                continue
            resets = (pos.diff().fillna(1) <= 0)
            doc_id = resets.cumsum()
            print(f"  Reconstructed document ids from '{c}' resets "
                  f"({doc_id.nunique()} inferred abstracts).")
            return doc_id

    if "FIELD" in df.columns:
        field = df["FIELD"].astype(str)
        prev_field = field.shift(1)
        is_boundary = field.isin(["BACKGROUND", "OBJECTIVE"]) & (~prev_field.isin(["BACKGROUND", "OBJECTIVE"]))
        if len(is_boundary) > 0:
            is_boundary.iloc[0] = True
        doc_id = is_boundary.cumsum()
        print(f"  WARNING: no id/position column found. Falling back to a "
              f"label-sequence HEURISTIC to infer abstract boundaries "
              f"({doc_id.nunique()} inferred abstracts). This is an "
              f"approximation, not ground truth.")
        return doc_id

    return None


In [ ]:
def add_sentence_context(df: pd.DataFrame, text_col: str = "TEXT", group_col: str = "PMID") -> pd.DataFrame:
    df = df.copy().reset_index(drop=True)

    if group_col not in df.columns:
        raise ValueError(
            f"add_sentence_context: '{group_col}' column is required and was "
            f"not found. Call infer_document_ids() first, or skip context "
            f"augmentation for this split -- do not fall back to an "
            f"ungrouped shift."
        )

    grouped = df.groupby(group_col, sort=False)[text_col]
    prev_text = grouped.shift(1).fillna("")
    next_text = grouped.shift(-1).fillna("")

    df[text_col] = prev_text + " [SEP] " + df[text_col] + " [SEP] " + next_text
    return df


In [ ]:
try:
    import spacy
    nlp = spacy.load("en_core_sci_md", disable=["tagger", "parser", "ner", "lemmatizer"])
    if "senter" not in nlp.pipe_names:
        nlp.add_pipe("sentencizer")

    def sentence_splitter(text: str) -> List[str]:
        doc = nlp(text)
        return [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 3]

    print("Using SciSpaCy sentence splitter.")
except Exception:
    print("SciSpaCy not available - using regex splitter.")

    def sentence_splitter(text: str) -> List[str]:
        sents = re.split(r'(?<=[.!?])\s+', str(text))
        return [s.strip() for s in sents if len(s.strip().split()) > 3]


In [ ]:
TRAIN_CSV = os.environ.get("TRAIN_CSV", "/content/drive/MyDrive/pubmed/PubMed_20k_RCT/pubmed_train_flat.csv")
VAL_CSV = os.environ.get("VAL_CSV", "/content/drive/MyDrive/pubmed/PubMed_20k_RCT/pubmed_val_flat.csv")
TEST_CSV = os.environ.get("TEST_CSV", "/content/drive/MyDrive/pubmed/PubMed_20k_RCT/pubmed_test_flat.csv")
UNLAB_CSV = os.environ.get("UNLAB_CSV", "/content/drive/MyDrive/pubmed/pubmed_abstracts_extracted_9996.csv")

In [ ]:
def safe_read_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        print(f"CSV path not found: {path} - returning empty DataFrame")
        return pd.DataFrame()
    return pd.read_csv(path)

In [ ]:
def extract_sentences_from_unlabeled(df: pd.DataFrame, col_name: str = "Abstract") -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame({"TEXT": [], "PMID": []})

    texts = df[col_name].fillna("").astype(str).tolist()
    all_sents, all_pmids = [], []
    for doc_idx, t in enumerate(texts):
        sents = sentence_splitter(t)
        sents = [clean_text_for_plm(s) for s in sents if len(s.strip()) > 0]
        all_sents.extend(sents)
        all_pmids.extend([f"unlab_{doc_idx}"] * len(sents))
    print(f"Collected {len(all_sents)} sentences from unlabeled corpus.")
    return pd.DataFrame({"TEXT": all_sents, "PMID": all_pmids})


In [ ]:
def fit_label_encoder_on_union(train, val, test, field_col="FIELD"):
    series_parts = []
    for d in [train, val, test]:
        if d is None or d.empty:
            continue
        if field_col in d.columns:
            series_parts.append(d[field_col].astype(str))
    if len(series_parts) == 0:
        return None
    union = pd.concat(series_parts).unique().tolist()
    le = LabelEncoder()
    le.fit(union)
    return le


In [ ]:
def safe_label_transform(df, le, field_col="FIELD", out_col="FIELD_ID"):
    if df is None or df.empty:
        return df
    df = df.copy()
    df[out_col] = -1
    mask_ok = df[field_col].astype(str).isin(le.classes_)
    df.loc[mask_ok, out_col] = le.transform(df.loc[mask_ok, field_col].astype(str))
    return df


In [ ]:
def compute_class_weights(train_df: pd.DataFrame, field_col: str = "FIELD_ID", beta: float = 0.9999):
    counts = train_df[field_col].value_counts().sort_index().values.astype(np.float64)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / np.maximum(effective_num, 1e-12)
    weights = weights / weights.sum() * len(counts)
    return torch.tensor(weights, dtype=torch.float)

In [ ]:
def focal_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    class_weights: Optional[torch.Tensor] = None,
    gamma: float = FOCAL_GAMMA,
    reduction: str = "mean"
) -> torch.Tensor:
    log_probs = F.log_softmax(logits, dim=-1)
    probs = log_probs.exp()

    target_log_probs = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
    target_probs = probs.gather(1, targets.unsqueeze(1)).squeeze(1)

    focal_term = (1.0 - target_probs).clamp(min=0.0) ** gamma
    loss = -focal_term * target_log_probs

    if class_weights is not None:
        alpha = class_weights.to(logits.device)[targets]
        loss = loss * alpha

    if reduction == "mean":
        return loss.mean()
    elif reduction == "sum":
        return loss.sum()
    elif reduction == "none":
        return loss
    else:
        raise ValueError(f"Unknown reduction: {reduction}")


In [ ]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts: List[str], labels: Optional[List[int]] = None):
        self.texts = ["" if (t is None or (isinstance(t, float) and np.isnan(t))) else str(t)
                      for t in texts]
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {"text": self.texts[idx]}
        if self.labels is not None:
            item["label"] = int(self.labels[idx])
        return item


In [ ]:
def make_collate_fn(tokenizer, max_length: int = MAX_LENGTH):
    def collate(batch: List[Dict]):
        texts = [b["text"] for b in batch]
        enc = tokenizer(
            texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt"
        )
        out = {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}
        if "label" in batch[0]:
            out["labels"] = torch.tensor([b["label"] for b in batch], dtype=torch.long)
        return out
    return collate

In [ ]:
def dataloader_from_df(
    df: pd.DataFrame,
    text_col: str = "TEXT",
    label_col: Optional[str] = None,
    tokenizer=None,
    batch_size: int = 32,
    shuffle: bool = False,
    max_length: int = MAX_LENGTH
) -> Optional[DataLoader]:
    assert tokenizer is not None
    if df is None or (isinstance(df, pd.DataFrame) and df.empty):
        return None

    texts = df[text_col].tolist() if isinstance(df, pd.DataFrame) else list(df)
    labels = df[label_col].tolist() if (label_col is not None and isinstance(df, pd.DataFrame)) else None

    dataset = TextClassificationDataset(texts, labels=labels)
    sampler = RandomSampler(dataset) if shuffle else SequentialSampler(dataset)
    collate_fn = make_collate_fn(tokenizer, max_length=max_length)

    return DataLoader(
        dataset, sampler=sampler, batch_size=batch_size, drop_last=False,
        pin_memory=(DEVICE.type == "cuda"), collate_fn=collate_fn
    )


In [ ]:
class InfiniteLoader:
    def __init__(self, loader: DataLoader):
        self.loader = loader
        self._iter = iter(self.loader)

    def __next__(self):
        try:
            return next(self._iter)
        except StopIteration:
            self._iter = iter(self.loader)
            return next(self._iter)


In [ ]:
def get_base_transformer(model: nn.Module) -> nn.Module:
    for attr in ["bert", "roberta", "electra", "base_model", "model", "transformer", "encoder"]:
        if hasattr(model, attr):
            return getattr(model, attr)
    return model


In [ ]:
def extract_base_encoder_state(model: nn.Module) -> Dict[str, torch.Tensor]:
    return get_base_transformer(model).state_dict()

In [ ]:
def transfer_encoder_weights(src_state: Dict[str, torch.Tensor], target_model: nn.Module) -> int:
    dest = get_base_transformer(target_model)
    dest_state = dest.state_dict()
    matched = {k: v for k, v in src_state.items() if k in dest_state and v.shape == dest_state[k].shape}
    if matched:
        dest_state.update(matched)
        dest.load_state_dict(dest_state, strict=False)
    return len(matched)

In [ ]:
class SimCSEModel(nn.Module):
    def __init__(self, base_model_name: str, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        cls = outputs.last_hidden_state[:, 0]
        return self.dropout(cls)

In [ ]:
def simcse_loss(e1: torch.Tensor, e2: torch.Tensor, temp: float = 0.05) -> torch.Tensor:
    device = e1.device
    e1 = F.normalize(e1, dim=1)
    e2 = F.normalize(e2, dim=1)
    logits = torch.matmul(e1, e2.T) / temp
    labels_idx = torch.arange(e1.size(0), device=device)
    return F.cross_entropy(logits, labels_idx)

In [ ]:
def pretrain_simcse(
    model: nn.Module, loader: DataLoader, device: torch.device = DEVICE,
    epochs: int = PRETRAIN_EPOCHS, lr: float = PRETRAIN_LR, temp: float = PRETRAIN_TEMP
):
    if loader is None or len(loader) == 0:
        print("No unlabeled loader - skipping SimCSE pretraining.")
        return

    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scaler = amp.GradScaler() if device.type == "cuda" else None

    for epoch in range(epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        pbar = tqdm(loader, desc=f"SimCSE epoch {epoch + 1}")

        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            optimizer.zero_grad(set_to_none=True)

            doubled_ids = torch.cat([input_ids, input_ids], dim=0)
            doubled_mask = torch.cat([attention_mask, attention_mask], dim=0)

            if scaler:
                with amp.autocast(device_type="cuda"):
                    emb = model(input_ids=doubled_ids, attention_mask=doubled_mask)
                    e1, e2 = emb.chunk(2, dim=0)
                    loss = simcse_loss(e1, e2, temp=temp)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                emb = model(input_ids=doubled_ids, attention_mask=doubled_mask)
                e1, e2 = emb.chunk(2, dim=0)
                loss = simcse_loss(e1, e2, temp=temp)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            loss_value = loss.detach().float().item()
            total_loss += loss_value
            n_batches += 1
            pbar.set_postfix({"loss": f"{loss_value:.4f}"})

        print(f"[SimCSE] Epoch {epoch + 1}/{epochs} - avg loss: {total_loss / max(1, n_batches):.4f}")
        if device.type == "cuda":
            torch.cuda.empty_cache()


In [ ]:
class RawTextDataset(Dataset):
    def __init__(self, texts: List[str], tokenizer, max_length: int = MAX_LENGTH):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_length)


In [ ]:
def mlm_dapt_pretrain(
    encoder_state: Dict[str, torch.Tensor],
    texts: List[str],
    tokenizer,
    base_model: str = BASE_MODEL,
    device: torch.device = DEVICE,
    epochs: int = DAPT_EPOCHS,
    lr: float = DAPT_LR,
    batch_size: int = DAPT_BATCH_SIZE,
    mlm_probability: float = DAPT_MLM_PROBABILITY,
    max_length: int = MAX_LENGTH
) -> Dict[str, torch.Tensor]:
    print(f"DAPT: continuing MLM pretraining on {len(texts)} unlabeled sentences "
          f"for {epochs} epoch(s)...")

    mlm_model = AutoModelForMaskedLM.from_pretrained(base_model)
    copied = transfer_encoder_weights(encoder_state, mlm_model)
    print(f"DAPT: transferred {copied} encoder tensors into the MLM model before continued pretraining.")
    mlm_model.to(device)

    dataset = RawTextDataset(texts, tokenizer, max_length=max_length)
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=mlm_probability)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collator)

    optimizer = optim.AdamW(mlm_model.parameters(), lr=lr)
    scaler = amp.GradScaler() if device.type == "cuda" else None

    for epoch in range(epochs):
        mlm_model.train()
        total_loss, n_batches = 0.0, 0
        pbar = tqdm(loader, desc=f"DAPT MLM epoch {epoch + 1}/{epochs}")

        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            if scaler:
                with amp.autocast(device_type="cuda"):
                    out = mlm_model(**batch)
                    loss = out.loss
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                out = mlm_model(**batch)
                loss = out.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), 1.0)
                optimizer.step()

            loss_value = loss.detach().float().item()
            total_loss += loss_value
            n_batches += 1
            pbar.set_postfix({"mlm_loss": f"{loss_value:.4f}"})

        print(f"[DAPT MLM] Epoch {epoch + 1}/{epochs} - avg loss: {total_loss / max(1, n_batches):.4f}")
        if device.type == "cuda":
            torch.cuda.empty_cache()

    dapt_state = extract_base_encoder_state(mlm_model)
    del mlm_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return dapt_state


In [ ]:
def fine_tune_multiple_lrs(
    encoder_state: Optional[Dict[str, torch.Tensor]],
    base_model: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: Optional[DataLoader] = None,
    learning_rates: List[float] = FINETUNE_LRS,
    epochs: int = FINETUNE_EPOCHS,
    patience: int = FINETUNE_PATIENCE,
    model_dir: str = SAVE_DIR,
    device: torch.device = DEVICE,
    train_df: pd.DataFrame = None,
    checkpoint_prefix: str = "finetuned"
):
    os.makedirs(model_dir, exist_ok=True)
    assert train_df is not None, "train_df must be provided to compute class weights."

    num_labels = int(train_df['FIELD_ID'].max()) + 1
    class_weights = compute_class_weights(train_df).to(device)

    def loss_fn(logits, targets):
        return focal_loss(logits, targets, class_weights=class_weights, gamma=FOCAL_GAMMA, reduction="mean")

    best_overall_f1 = 0.0
    best_model_path = None
    test_micro_f1 = 0.0

    for lr in learning_rates:
        print(f"\n--- Training with lr={lr} ---")

        clf = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=num_labels)
        if encoder_state is not None:
            copied = transfer_encoder_weights(encoder_state, clf)
            print(f"Transferred {copied} encoder tensors into classifier (best-effort).")
        clf.to(device)

        optimizer = optim.AdamW(clf.parameters(), lr=lr)
        num_training_steps = epochs * max(1, len(train_loader))
        lr_scheduler = get_scheduler(
            "linear", optimizer=optimizer,
            num_warmup_steps=int(0.1 * num_training_steps),
            num_training_steps=num_training_steps
        )
        scaler = amp.GradScaler() if device.type == "cuda" else None

        best_val_f1 = 0.0
        patience_counter = 0
        local_best_path = os.path.join(model_dir, f"{checkpoint_prefix}_lr{lr:.0e}.pt")

        for epoch in range(epochs):
            clf.train()
            total_loss, n_batches = 0.0, 0

            for batch in tqdm(train_loader, desc=f"Train epoch {epoch + 1}/{epochs} (lr={lr})"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                optimizer.zero_grad()

                if scaler:
                    with amp.autocast(device_type="cuda"):
                        outputs = clf(input_ids=input_ids, attention_mask=attention_mask)
                        loss = loss_fn(outputs.logits, labels)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = clf(input_ids=input_ids, attention_mask=attention_mask)
                    loss = loss_fn(outputs.logits, labels)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
                    optimizer.step()

                lr_scheduler.step()
                total_loss += float(loss.detach().cpu().item())
                n_batches += 1

            print(f"[LR {lr}] Epoch {epoch + 1} - avg_train_loss: {total_loss / max(1, n_batches):.4f}")

            clf.eval()
            val_preds, val_labels = [], []
            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    outputs = clf(input_ids=input_ids, attention_mask=attention_mask)
                    preds = torch.argmax(outputs.logits, dim=-1)
                    val_preds.extend(preds.cpu().numpy())
                    val_labels.extend(batch["labels"].numpy())

            micro_f1 = f1_score(val_labels, val_preds, average="micro") if val_labels else 0.0
            print(f"[LR {lr}] Epoch {epoch + 1} - val micro-F1: {micro_f1:.4f}")

            if micro_f1 > best_val_f1:
                best_val_f1 = micro_f1
                patience_counter = 0
                torch.save({'model_state': clf.state_dict(), 'val_f1': best_val_f1, 'lr': lr, 'epoch': epoch},
                           local_best_path)
                print(f"  Saved new best for lr {lr} -> {local_best_path}")
            else:
                patience_counter += 1
                print(f"  No improvement, patience {patience_counter}/{patience}")
                if patience_counter >= patience:
                    print("  Early stopping for this lr.")
                    break

        if best_val_f1 > best_overall_f1:
            best_overall_f1 = best_val_f1
            best_model_path = local_best_path

        del clf, optimizer, lr_scheduler
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if best_model_path is None:
        raise RuntimeError("No best model saved during fine-tuning.")

    best_ckpt = torch.load(best_model_path, map_location=device)
    best_model = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=num_labels)
    best_model.load_state_dict(best_ckpt['model_state'], strict=False)
    best_model.to(device)
    best_model.eval()

    if test_loader is not None:
        test_preds, test_labels = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=-1)
                test_preds.extend(preds.cpu().numpy())
                test_labels.extend(batch["labels"].numpy())
        test_micro_f1 = f1_score(test_labels, test_preds, average="micro")
        print(f"[{checkpoint_prefix}] Test micro-F1 (raw, pre-ablation-report): {test_micro_f1:.4f}")

    return best_model, best_overall_f1, test_micro_f1


In [ ]:
def fit_temperature_scaling(
    model: nn.Module, val_loader: DataLoader, device: torch.device = DEVICE,
    max_iter: int = 50, init_T: float = 1.5
) -> float:
    model.eval()
    logits_list, labels_list = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            logits_list.append(out.logits.detach().cpu())
            labels_list.append(batch["labels"])

    logits = torch.cat(logits_list, dim=0).to(device)
    labels = torch.cat(labels_list, dim=0).to(device)

    log_T = torch.nn.Parameter(torch.log(torch.tensor(init_T, device=device)))
    optimizer = optim.LBFGS([log_T], lr=0.01, max_iter=max_iter)
    nll_fn = nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        T = log_T.exp().clamp(min=0.5, max=5.0)
        loss = nll_fn(logits / T, labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    T_final = float(log_T.exp().clamp(min=0.5, max=5.0).detach().cpu().item())
    print(f"Fitted temperature scaling: T = {T_final:.4f}")
    return T_final


In [ ]:
def soft_distillation_finetune(
    student: nn.Module,
    teacher: nn.Module,
    teacher_temperature: float,
    labeled_loader: DataLoader,
    unlabeled_loader: DataLoader,
    val_loader: DataLoader,
    train_df: pd.DataFrame,
    epochs: int = SOFT_DISTILL_EPOCHS,
    lr: float = SOFT_DISTILL_LR,
    patience: int = SOFT_DISTILL_PATIENCE,
    distill_weight: float = SOFT_DISTILL_WEIGHT,
    device: torch.device = DEVICE,
    checkpoint_path: Optional[str] = None
) -> Tuple[nn.Module, float]:
   
    class_weights = compute_class_weights(train_df).to(device)

    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False

    optimizer = optim.AdamW(student.parameters(), lr=lr)
    num_training_steps = epochs * max(1, len(labeled_loader))
    lr_scheduler = get_scheduler(
        "linear", optimizer=optimizer,
        num_warmup_steps=int(0.1 * num_training_steps),
        num_training_steps=num_training_steps
    )
    scaler = amp.GradScaler() if device.type == "cuda" else None

    unlabeled_iter = InfiniteLoader(unlabeled_loader)

    best_state = None
    best_val_f1 = -1.0
    patience_counter = 0

    for epoch in range(epochs):
        student.train()
        total_loss, total_sup, total_distill, n_batches = 0.0, 0.0, 0.0, 0

        for batch in tqdm(labeled_loader, desc=f"Soft distill epoch {epoch + 1}/{epochs}"):
            u_batch = next(unlabeled_iter)

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            u_ids = u_batch["input_ids"].to(device)
            u_mask = u_batch["attention_mask"].to(device)

            optimizer.zero_grad(set_to_none=True)

            if scaler:
                with amp.autocast(device_type="cuda"):
                    sup_logits = student(input_ids=input_ids, attention_mask=attention_mask).logits
                    sup_loss = focal_loss(sup_logits, labels, class_weights=class_weights,
                                           gamma=FOCAL_GAMMA, reduction="mean")

                    student_u_logits = student(input_ids=u_ids, attention_mask=u_mask).logits
                    with torch.no_grad():
                        teacher_u_logits = teacher(input_ids=u_ids, attention_mask=u_mask).logits
                        teacher_probs = F.softmax(teacher_u_logits / teacher_temperature, dim=-1)
                    student_log_probs = F.log_softmax(student_u_logits, dim=-1)
                    distill_loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean")

                    loss = sup_loss + distill_weight * distill_loss

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                sup_logits = student(input_ids=input_ids, attention_mask=attention_mask).logits
                sup_loss = focal_loss(sup_logits, labels, class_weights=class_weights,
                                       gamma=FOCAL_GAMMA, reduction="mean")

                student_u_logits = student(input_ids=u_ids, attention_mask=u_mask).logits
                with torch.no_grad():
                    teacher_u_logits = teacher(input_ids=u_ids, attention_mask=u_mask).logits
                    teacher_probs = F.softmax(teacher_u_logits / teacher_temperature, dim=-1)
                student_log_probs = F.log_softmax(student_u_logits, dim=-1)
                distill_loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean")

                loss = sup_loss + distill_weight * distill_loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
                optimizer.step()

            lr_scheduler.step()
            total_loss += float(loss.detach().cpu().item())
            total_sup += float(sup_loss.detach().cpu().item())
            total_distill += float(distill_loss.detach().cpu().item())
            n_batches += 1

        print(f"[Soft distill] Epoch {epoch + 1} - avg_loss: {total_loss / max(1, n_batches):.4f} "
              f"(sup: {total_sup / max(1, n_batches):.4f}, "
              f"distill: {total_distill / max(1, n_batches):.4f})")

        student.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for vb in val_loader:
                vi = vb["input_ids"].to(device)
                vm = vb["attention_mask"].to(device)
                o = student(input_ids=vi, attention_mask=vm)
                p = torch.argmax(o.logits, dim=-1)
                val_preds.extend(p.cpu().numpy())
                val_labels.extend(vb["labels"].numpy())

        val_f1 = f1_score(val_labels, val_preds, average="micro")
        print(f"[Soft distill] Epoch {epoch + 1} - val micro-F1: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
            patience_counter = 0
            if checkpoint_path is not None:
                torch.save({'model_state': best_state, 'val_f1': best_val_f1}, checkpoint_path)
                print(f"  Saved new best Config 3 checkpoint -> {checkpoint_path}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping soft distillation.")
                break

    if best_state is not None:
        student.load_state_dict(best_state)

    return student, best_val_f1


In [ ]:
def supervised_contrastive_loss(
    embeddings: torch.Tensor, labels: torch.Tensor, temperature: float = SUPCON_TEMPERATURE
) -> torch.Tensor:
    device = embeddings.device
    embeddings = F.normalize(embeddings, dim=1)
    batch_size = embeddings.size(0)

    sim_matrix = torch.matmul(embeddings, embeddings.T) / temperature
    sim_max, _ = sim_matrix.max(dim=1, keepdim=True)
    sim_matrix = sim_matrix - sim_max.detach()

    labels = labels.view(-1, 1)
    same_class_mask = torch.eq(labels, labels.T).float().to(device)
    self_mask = torch.eye(batch_size, device=device)
    positive_mask = same_class_mask * (1.0 - self_mask)

    exp_sim = torch.exp(sim_matrix) * (1.0 - self_mask)
    log_prob = sim_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-12)

    pos_counts = positive_mask.sum(dim=1)
    has_positive = (pos_counts > 0).float()

    mean_log_prob_pos = (positive_mask * log_prob).sum(dim=1) / (pos_counts + 1e-12)
    loss = -(mean_log_prob_pos * has_positive).sum() / (has_positive.sum() + 1e-12)
    return loss


In [ ]:
class SupConClassifierModel(nn.Module):
    class Output:
        __slots__ = ("logits", "projection")

    def __init__(self, base_model_name: str, num_labels: int,
                 projection_dim: int = SUPCON_PROJECTION_DIM, dropout: float = 0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.projection = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, projection_dim)
        )

    def forward(self, input_ids, attention_mask, return_projection: bool = False):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        cls_repr = outputs.last_hidden_state[:, 0]

        out = SupConClassifierModel.Output()
        out.logits = self.classifier(self.dropout(cls_repr))
        out.projection = self.projection(cls_repr) if return_projection else None
        return out


In [ ]:
def transfer_full_classifier_weights(src_model: nn.Module, dst_model: nn.Module) -> None:
    n_encoder = transfer_encoder_weights(extract_base_encoder_state(src_model), dst_model)

    src_clf = getattr(src_model, "classifier", None)
    dst_clf = getattr(dst_model, "classifier", None)
    n_clf = 0
    if src_clf is not None and dst_clf is not None:
        src_state = src_clf.state_dict()
        dst_state = dst_clf.state_dict()
        matched = {k: v for k, v in src_state.items() if k in dst_state and v.shape == dst_state[k].shape}
        if matched:
            dst_state.update(matched)
            dst_clf.load_state_dict(dst_state, strict=False)
            n_clf = len(matched)

    print(f"Transferred {n_encoder} encoder tensors + {n_clf} classifier-head tensors "
          f"from {type(src_model).__name__} into {type(dst_model).__name__}.")


In [ ]:
def supcon_finetune(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    train_df: pd.DataFrame,
    epochs: int = SUPCON_EPOCHS,
    lr: float = SUPCON_LR,
    patience: int = SUPCON_PATIENCE,
    supcon_weight: float = SUPCON_WEIGHT,
    supcon_temperature: float = SUPCON_TEMPERATURE,
    warmup_fraction: float = SUPCON_WARMUP_FRACTION,
    device: torch.device = DEVICE,
    checkpoint_path: Optional[str] = None
) -> Tuple[nn.Module, float]:
  
    class_weights = compute_class_weights(train_df).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    num_training_steps = epochs * max(1, len(train_loader))
    warmup_steps = max(1, int(warmup_fraction * num_training_steps))
    lr_scheduler = get_scheduler(
        "linear", optimizer=optimizer,
        num_warmup_steps=int(0.1 * num_training_steps),
        num_training_steps=num_training_steps
    )
    scaler = amp.GradScaler() if device.type == "cuda" else None

    best_state = None
    best_val_f1 = -1.0
    patience_counter = 0
    global_step = 0

    for epoch in range(epochs):
        model.train()
        total_loss, total_focal, total_supcon, total_eff_weight, n_batches = 0.0, 0.0, 0.0, 0.0, 0

        for batch in tqdm(train_loader, desc=f"SupCon epoch {epoch + 1}/{epochs}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            effective_supcon_weight = supcon_weight * min(1.0, global_step / warmup_steps)

            optimizer.zero_grad(set_to_none=True)

            if scaler:
                with amp.autocast(device_type="cuda"):
                    out = model(input_ids=input_ids, attention_mask=attention_mask, return_projection=True)
                    focal = focal_loss(out.logits, labels, class_weights=class_weights,
                                        gamma=FOCAL_GAMMA, reduction="mean")
                    supcon = supervised_contrastive_loss(out.projection, labels, temperature=supcon_temperature)
                    loss = focal + effective_supcon_weight * supcon
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(input_ids=input_ids, attention_mask=attention_mask, return_projection=True)
                focal = focal_loss(out.logits, labels, class_weights=class_weights,
                                    gamma=FOCAL_GAMMA, reduction="mean")
                supcon = supervised_contrastive_loss(out.projection, labels, temperature=supcon_temperature)
                loss = focal + effective_supcon_weight * supcon
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            lr_scheduler.step()
            global_step += 1
            total_loss += float(loss.detach().cpu().item())
            total_focal += float(focal.detach().cpu().item())
            total_supcon += float(supcon.detach().cpu().item())
            total_eff_weight += effective_supcon_weight
            n_batches += 1

        print(f"[SupCon] Epoch {epoch + 1} - avg_loss: {total_loss / max(1, n_batches):.4f} "
              f"(focal: {total_focal / max(1, n_batches):.4f}, "
              f"supcon: {total_supcon / max(1, n_batches):.4f}, "
              f"avg_effective_supcon_weight: {total_eff_weight / max(1, n_batches):.4f})")

        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for vb in val_loader:
                vi = vb["input_ids"].to(device)
                vm = vb["attention_mask"].to(device)
                o = model(input_ids=vi, attention_mask=vm, return_projection=False)
                p = torch.argmax(o.logits, dim=-1)
                val_preds.extend(p.cpu().numpy())
                val_labels.extend(vb["labels"].numpy())

        val_f1 = f1_score(val_labels, val_preds, average="micro")
        val_macro_f1 = f1_score(val_labels, val_preds, average="macro")
        print(f"[SupCon] Epoch {epoch + 1} - val micro-F1: {val_f1:.4f}, val macro-F1: {val_macro_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            if checkpoint_path is not None:
                torch.save({'model_state': best_state, 'val_f1': best_val_f1}, checkpoint_path)
                print(f"  Saved new best Config 4 checkpoint -> {checkpoint_path}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping SupCon fine-tuning.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_f1


In [ ]:
def bootstrap_metric_ci(
    y_true: List[int], y_pred: List[int], metric_fn, n_boot: int = BOOTSTRAP_N,
    ci: float = BOOTSTRAP_CI, seed: int = SEED
) -> Tuple[float, float, float]:
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)

    scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.randint(0, n, size=n)
        scores[i] = metric_fn(y_true[idx], y_pred[idx])

    alpha = (1 - ci) / 2
    lo, hi = np.quantile(scores, [alpha, 1 - alpha])
    return float(scores.mean()), float(lo), float(hi)


In [ ]:
def evaluate_on_test(
    model: nn.Module, test_loader: DataLoader, device: torch.device,
    config_name: str, class_names: Optional[List[str]] = None
) -> Dict:
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(out.logits, dim=-1)
            y_pred.extend(preds.cpu().numpy())
            y_true.extend(batch["labels"].numpy())

    print(f"\n=== {config_name}: test set classification report ===")
    print(classification_report(y_true, y_pred, target_names=class_names))

    micro_f1 = f1_score(y_true, y_pred, average="micro")
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    per_class_f1 = f1_score(y_true, y_pred, average=None)

    micro_mean, micro_lo, micro_hi = bootstrap_metric_ci(
        y_true, y_pred, lambda a, b: f1_score(a, b, average="micro"))
    macro_mean, macro_lo, macro_hi = bootstrap_metric_ci(
        y_true, y_pred, lambda a, b: f1_score(a, b, average="macro"))

    print(f"{config_name} - micro-F1: {micro_f1:.4f}  "
          f"(bootstrap {int(BOOTSTRAP_CI*100)}% CI [{micro_lo:.4f}, {micro_hi:.4f}], n_boot={BOOTSTRAP_N})")
    print(f"{config_name} - macro-F1: {macro_f1:.4f}  "
          f"(bootstrap {int(BOOTSTRAP_CI*100)}% CI [{macro_lo:.4f}, {macro_hi:.4f}], n_boot={BOOTSTRAP_N})")

    return {
        "config": config_name,
        "micro_f1": micro_f1, "micro_ci_lo": micro_lo, "micro_ci_hi": micro_hi,
        "macro_f1": macro_f1, "macro_ci_lo": macro_lo, "macro_ci_hi": macro_hi,
        "per_class_f1": per_class_f1.tolist(),
        "class_names": class_names,
        "y_true": y_true, "y_pred": y_pred,
    }


In [ ]:
def paired_bootstrap_diff_ci(
    y_true: List[int], y_pred_a: List[int], y_pred_b: List[int], metric_fn,
    n_boot: int = BOOTSTRAP_N, ci: float = BOOTSTRAP_CI, seed: int = SEED
) -> Tuple[float, float, float, bool]:
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    y_pred_a = np.asarray(y_pred_a)
    y_pred_b = np.asarray(y_pred_b)
    n = len(y_true)

    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.randint(0, n, size=n)
        diffs[i] = metric_fn(y_true[idx], y_pred_b[idx]) - metric_fn(y_true[idx], y_pred_a[idx])

    alpha = (1 - ci) / 2
    lo, hi = np.quantile(diffs, [alpha, 1 - alpha])
    significant = not (lo <= 0 <= hi)
    return float(diffs.mean()), float(lo), float(hi), significant


In [ ]:
def print_pairwise_significance(results: List[Dict]):
    if len(results) < 2:
        return

    print("\n" + "=" * 92)
    print(f"PAIRWISE SIGNIFICANCE (paired bootstrap on the SAME test set, "
          f"{int(BOOTSTRAP_CI*100)}% CI, n_boot={BOOTSTRAP_N})")
    print("=" * 92)

    baseline = results[0]
    pairs = []
    for r in results[1:]:
        pairs.append(("vs. " + baseline["config"], baseline, r))
    for i in range(1, len(results) - 1):
        pairs.append(("vs. previous config", results[i], results[i + 1]))

    rows = []
    for label, r_a, r_b in pairs:
        mean_diff, lo, hi, sig = paired_bootstrap_diff_ci(
            r_a["y_true"], r_a["y_pred"], r_b["y_pred"],
            lambda a, b: f1_score(a, b, average="micro")
        )
        marker = " *" if sig else ""
        print(f"{r_b['config']:<32} {label:<24} \u0394micro-F1={mean_diff:+.4f}  "
              f"90% CI=[{lo:+.4f}, {hi:+.4f}]{marker}")
        rows.append({
            "config_b": r_b["config"], "config_a": r_a["config"],
            "delta_micro_f1": mean_diff, "ci_lo": lo, "ci_hi": hi, "significant": sig
        })

    print("(* = 90% CI excludes zero, i.e. statistically significant improvement)")
    print("=" * 92)
    return rows


In [ ]:
def print_ablation_table(results: List[Dict]):
    print("\n" + "=" * 92)
    print(f"ABLATION SUMMARY (bootstrap {int(BOOTSTRAP_CI*100)}% CI, n_boot={BOOTSTRAP_N})")
    print("=" * 92)
    print(f"{'Config':<32} {'Micro-F1':>10} {'90% CI':>18} {'Macro-F1':>10} {'90% CI':>18}")
    for r in results:
        micro_ci = f"[{r['micro_ci_lo']:.4f}, {r['micro_ci_hi']:.4f}]"
        macro_ci = f"[{r['macro_ci_lo']:.4f}, {r['macro_ci_hi']:.4f}]"
        print(f"{r['config']:<32} {r['micro_f1']:.4f}     {micro_ci:>18} {r['macro_f1']:.4f}     {macro_ci:>18}")
    print("=" * 92)
    print("NOTE: the CI overlap/non-overlap between rows above is NOT a valid")
    print("significance test for comparing configs against each other (they share")
    print("the same test set, so their sampling noise is correlated). See the")
    print("PAIRWISE SIGNIFICANCE table below for the correct paired test.")


In [ ]:
def save_ablation_results(results: List[Dict], out_path: str):
    class_names = results[0]["class_names"] if results and results[0]["class_names"] else None
    fieldnames = ["config", "micro_f1", "micro_ci_lo", "micro_ci_hi", "macro_f1", "macro_ci_lo", "macro_ci_hi"]
    if class_names:
        fieldnames += [f"f1_{c}" for c in class_names]

    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            row = {k: r[k] for k in fieldnames if k in r}
            if class_names:
                for c, v in zip(class_names, r["per_class_f1"]):
                    row[f"f1_{c}"] = v
            writer.writerow(row)
    print(f"Saved ablation results to {out_path}")


In [ ]:
def save_pairwise_significance(rows: Optional[List[Dict]], out_path: str):
    if not rows:
        return
    fieldnames = ["config_b", "config_a", "delta_micro_f1", "ci_lo", "ci_hi", "significant"]
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)
    print(f"Saved pairwise significance results to {out_path}")


In [ ]:
def try_load_best_finetune_checkpoint(
    base_model: str, num_labels: int, model_dir: str, checkpoint_prefix: str,
    learning_rates: List[float], device: torch.device
) -> Tuple[Optional[nn.Module], Optional[float]]:
    best_path, best_val_f1 = None, -1.0
    for lr in learning_rates:
        path = os.path.join(model_dir, f"{checkpoint_prefix}_lr{lr:.0e}.pt")
        if os.path.exists(path):
            ckpt = torch.load(path, map_location=device)
            if ckpt.get("val_f1", -1.0) > best_val_f1:
                best_val_f1 = ckpt["val_f1"]
                best_path = path

    if best_path is None:
        return None, None

    ckpt = torch.load(best_path, map_location=device)
    model = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=num_labels)
    model.load_state_dict(ckpt["model_state"], strict=False)
    model.to(device)
    model.eval()
    print(f"RESUMED '{checkpoint_prefix}' from {best_path} "
          f"(recorded val micro-F1={best_val_f1:.4f}) -- skipping retraining for this config.")
    return model, best_val_f1


In [ ]:
def try_load_checkpoint(
    checkpoint_path: str, model_builder_fn, device: torch.device, label: str = "checkpoint"
) -> Tuple[Optional[nn.Module], Optional[float]]:
    
    if not os.path.exists(checkpoint_path):
        return None, None
    ckpt = torch.load(checkpoint_path, map_location=device)
    model = model_builder_fn()
    model.load_state_dict(ckpt["model_state"], strict=False)
    model.to(device)
    model.eval()
    val_f1 = ckpt.get("val_f1", None)
    print(f"RESUMED '{label}' from {checkpoint_path} "
          f"(recorded val micro-F1={val_f1:.4f}) -- skipping retraining for this config.")
    return model, val_f1


In [ ]:
def per_class_paired_significance(
    result_a: Dict, result_b: Dict, class_names: List[str],
    n_boot: int = BOOTSTRAP_N, ci: float = BOOTSTRAP_CI, seed: int = SEED
) -> List[Dict]:
    
    print(f"\n{'=' * 92}")
    print(f"PER-CLASS SIGNIFICANCE: {result_b['config']} vs {result_a['config']} "
          f"(paired bootstrap, {int(ci*100)}% CI, n_boot={n_boot})")
    print("=" * 92)
    print(f"{'Class':<14} {'Delta F1':>10} {'90% CI':>20}")

    rows = []
    for class_id, cname in enumerate(class_names):
        def metric_fn(a, b, cid=class_id):
            return f1_score(a, b, labels=[cid], average="micro", zero_division=0)

        mean_diff, lo, hi, sig = paired_bootstrap_diff_ci(
            result_a["y_true"], result_a["y_pred"], result_b["y_pred"], metric_fn,
            n_boot=n_boot, ci=ci, seed=seed
        )
        marker = " *" if sig else ""
        print(f"{cname:<14} {mean_diff:+.4f}     [{lo:+.4f}, {hi:+.4f}]{marker}")
        rows.append({
            "class": cname, "config_a": result_a["config"], "config_b": result_b["config"],
            "delta_f1": mean_diff, "ci_lo": lo, "ci_hi": hi, "significant": sig
        })

    print("(* = 90% CI excludes zero, i.e. statistically significant per-class difference)")
    print("=" * 92)
    return rows


In [ ]:
def save_per_class_significance(rows: Optional[List[Dict]], out_path: str):
    if not rows:
        return
    fieldnames = ["class", "config_a", "config_b", "delta_f1", "ci_lo", "ci_hi", "significant"]
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)
    print(f"Saved per-class significance results to {out_path}")

In [ ]:

def build_data():
    train_df = safe_read_csv(TRAIN_CSV)
    val_df = safe_read_csv(VAL_CSV)
    test_df = safe_read_csv(TEST_CSV)
    unlab_df = safe_read_csv(UNLAB_CSV)

    print(f"Train: {len(train_df)}")
    print(f"Val: {len(val_df)}")
    print(f"Test: {len(test_df)}")
    print(f"Unlabeled: {len(unlab_df)}")

    if not unlab_df.empty:
        unlab_df = unlab_df.sample(n=min(5000, len(unlab_df)), random_state=SEED).reset_index(drop=True)
        print(f"Using {len(unlab_df)} unlabeled abstracts (sampled).")

    all_sentences_df = extract_sentences_from_unlabeled(unlab_df, col_name="Abstract")

    for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        if df is None or df.empty:
            continue

        if "TEXT" in df.columns:
            df["TEXT"] = df["TEXT"].apply(clean_text_for_plm)
        elif "text" in df.columns:
            df["TEXT"] = df["text"].apply(clean_text_for_plm)
        elif "abstract" in df.columns:
            df["TEXT"] = df["abstract"].apply(clean_text_for_plm)
        else:
            first_col = df.columns[0]
            df["TEXT"] = df[first_col].apply(clean_text_for_plm)

        print(f"Resolving document boundaries for '{name}' split...")
        doc_ids = infer_document_ids(df)
        if doc_ids is not None:
            df["PMID"] = doc_ids
            df_ctx = add_sentence_context(df, text_col="TEXT", group_col="PMID")
        else:
            print(f"WARNING: could not identify/reconstruct abstract boundaries for "
                  f"'{name}' split. Skipping prev/next-sentence context for this split.")
            df_ctx = df.copy()

        if name == "train":
            train_df = df_ctx
        elif name == "val":
            val_df = df_ctx
        else:
            test_df = df_ctx

    le = fit_label_encoder_on_union(train_df, val_df, test_df, field_col="FIELD")
    if le is not None:
        train_df = safe_label_transform(train_df, le)
        val_df = safe_label_transform(val_df, le)
        test_df = safe_label_transform(test_df, le)

    return train_df, val_df, test_df, unlab_df, all_sentences_df, le


In [ ]:

def run_ablation():
    train_df, val_df, test_df, unlab_df, all_sentences_df, le = build_data()
    class_names = le.classes_.tolist() if le is not None else None

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    train_df = train_df.dropna(subset=["FIELD_ID"])
    val_df_labeled = val_df.dropna(subset=["FIELD_ID"])
    test_df_labeled = test_df.dropna(subset=["FIELD_ID"])

    train_loader = dataloader_from_df(train_df, "TEXT", "FIELD_ID", tokenizer, FINETUNE_BATCH_SIZE, shuffle=True)
    val_loader = dataloader_from_df(val_df_labeled, "TEXT", "FIELD_ID", tokenizer, FINETUNE_BATCH_SIZE, shuffle=False)
    test_loader = dataloader_from_df(test_df_labeled, "TEXT", "FIELD_ID", tokenizer, FINETUNE_BATCH_SIZE, shuffle=False)

    if train_loader is None or val_loader is None or test_loader is None:
        print("Missing labeled data; cannot run ablation. Exiting.")
        return

    unlab_texts = all_sentences_df["TEXT"].tolist() if all_sentences_df is not None and not all_sentences_df.empty else []
    simcse_batch_size = max(1, min(PRETRAIN_BATCH_SIZE, len(unlab_texts))) if unlab_texts else PRETRAIN_BATCH_SIZE
    unlab_loader_simcse = dataloader_from_df(all_sentences_df, "TEXT", None, tokenizer, simcse_batch_size, shuffle=True)
    unlab_loader_distill = dataloader_from_df(all_sentences_df, "TEXT", None, tokenizer, FINETUNE_BATCH_SIZE, shuffle=True)

    results = []
    num_labels = int(train_df["FIELD_ID"].max()) + 1

    plm_tag = selected_model


    print("\n" + "#" * 78)
    print("# CONFIG 1: Fine-tuning only (SimCSE + PMID-safe context + focal loss)")
    print("#" * 78)
    config1_model, config1_val_f1 = try_load_best_finetune_checkpoint(
        base_model=BASE_MODEL, num_labels=num_labels, model_dir=SAVE_DIR,
        checkpoint_prefix=f"config1_finetune_{plm_tag}", learning_rates=FINETUNE_LRS, device=DEVICE
    )

    if config1_model is None:
        print("No existing Config 1 checkpoint found -- running SimCSE pretraining + fine-tuning from scratch.")
        simcse_encoder = SimCSEModel(BASE_MODEL, dropout=0.2)
        if unlab_loader_simcse is not None:
            pretrain_simcse(simcse_encoder, unlab_loader_simcse, device=DEVICE,
                             epochs=PRETRAIN_EPOCHS, lr=PRETRAIN_LR, temp=PRETRAIN_TEMP)
        else:
            print("No unlabeled sentences available; skipping SimCSE pretraining.")
        simcse_state = extract_base_encoder_state(simcse_encoder)
        del simcse_encoder
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        config1_model, config1_val_f1, _ = fine_tune_multiple_lrs(
            encoder_state=simcse_state, base_model=BASE_MODEL,
            train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
            train_df=train_df, learning_rates=FINETUNE_LRS, epochs=FINETUNE_EPOCHS,
            patience=FINETUNE_PATIENCE, model_dir=SAVE_DIR, checkpoint_prefix=f"config1_finetune_{plm_tag}"
        )
    else:
        simcse_state = None

    results.append(evaluate_on_test(config1_model, test_loader, DEVICE, "Config 1: Fine-tuning only", class_names))

    print("\n" + "#" * 78)
    print("# CONFIG 2: + MLM domain-adaptive pretraining (DAPT)")
    print("#" * 78)
    config2_model, config2_val_f1 = try_load_best_finetune_checkpoint(
        base_model=BASE_MODEL, num_labels=num_labels, model_dir=SAVE_DIR,
        checkpoint_prefix=f"config2_dapt_finetune_{plm_tag}", learning_rates=FINETUNE_LRS, device=DEVICE
    )

    if config2_model is None:
        print("No existing Config 2 checkpoint found -- running DAPT + fine-tuning from scratch.")
        if simcse_state is None:
            print("Re-running SimCSE pretraining (needed as DAPT's starting point)...")
            simcse_encoder = SimCSEModel(BASE_MODEL, dropout=0.2)
            if unlab_loader_simcse is not None:
                pretrain_simcse(simcse_encoder, unlab_loader_simcse, device=DEVICE,
                                 epochs=PRETRAIN_EPOCHS, lr=PRETRAIN_LR, temp=PRETRAIN_TEMP)
            simcse_state = extract_base_encoder_state(simcse_encoder)
            del simcse_encoder
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        if unlab_texts:
            dapt_state = mlm_dapt_pretrain(
                encoder_state=simcse_state, texts=unlab_texts, tokenizer=tokenizer,
                base_model=BASE_MODEL, device=DEVICE, epochs=DAPT_EPOCHS, lr=DAPT_LR,
                batch_size=DAPT_BATCH_SIZE, mlm_probability=DAPT_MLM_PROBABILITY, max_length=MAX_LENGTH
            )
        else:
            print("No unlabeled sentences available; skipping DAPT, reusing SimCSE encoder state.")
            dapt_state = simcse_state

        config2_model, config2_val_f1, _ = fine_tune_multiple_lrs(
            encoder_state=dapt_state, base_model=BASE_MODEL,
            train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
            train_df=train_df, learning_rates=FINETUNE_LRS, epochs=FINETUNE_EPOCHS,
            patience=FINETUNE_PATIENCE, model_dir=SAVE_DIR, checkpoint_prefix=f"config2_dapt_finetune_{plm_tag}"
        )

    results.append(evaluate_on_test(config2_model, test_loader, DEVICE, "Config 2: + MLM DAPT", class_names))

    del config1_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("\n" + "#" * 78)
    print("# CONFIG 3: + Soft distillation on unlabeled data (frozen calibrated teacher)")
    print("#" * 78)
    config3_ckpt_path = os.path.join(SAVE_DIR, f"config3_distill_{plm_tag}.pt")
    config3_model, config3_val_f1 = try_load_checkpoint(
        config3_ckpt_path,
        lambda: AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=num_labels),
        DEVICE, label="config3_distill"
    )

    if config3_model is None and unlab_loader_distill is not None:
        teacher_T = fit_temperature_scaling(config2_model, val_loader, device=DEVICE)

        student_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=num_labels)
        student_model.load_state_dict(config2_model.state_dict())
        student_model.to(DEVICE)

        config3_model, config3_val_f1 = soft_distillation_finetune(
            student=student_model, teacher=config2_model, teacher_temperature=teacher_T,
            labeled_loader=train_loader, unlabeled_loader=unlab_loader_distill, val_loader=val_loader,
            train_df=train_df, epochs=SOFT_DISTILL_EPOCHS, lr=SOFT_DISTILL_LR,
            patience=SOFT_DISTILL_PATIENCE, distill_weight=SOFT_DISTILL_WEIGHT, device=DEVICE,
            checkpoint_path=config3_ckpt_path
        )

    if config3_model is not None:
        results.append(evaluate_on_test(config3_model, test_loader, DEVICE,
                                         "Config 3: + Soft distillation", class_names))
    else:
        print("No unlabeled sentences available; skipping soft distillation (Config 3).")


    print("\n" + "#" * 78)
    print("# CONFIG 4: + Supervised Contrastive Learning (SupCon, on labeled data)")
    print("#" * 78)
    config4_ckpt_path = os.path.join(SAVE_DIR, f"config4_supcon_{plm_tag}.pt")
    config4_model, config4_val_f1 = try_load_checkpoint(
        config4_ckpt_path,
        lambda: SupConClassifierModel(BASE_MODEL, num_labels=num_labels, projection_dim=SUPCON_PROJECTION_DIM),
        DEVICE, label="config4_supcon"
    )

    if config4_model is None:
        supcon_base_model = config3_model if config3_model is not None else config2_model
        if config3_model is None:
            print("Config 3 was skipped; building Config 4 on top of Config 2 instead.")

        config4_model = SupConClassifierModel(BASE_MODEL, num_labels=num_labels, projection_dim=SUPCON_PROJECTION_DIM)
        transfer_full_classifier_weights(supcon_base_model, config4_model)
        config4_model.to(DEVICE)

        config4_model, config4_val_f1 = supcon_finetune(
            model=config4_model, train_loader=train_loader, val_loader=val_loader, train_df=train_df,
            epochs=SUPCON_EPOCHS, lr=SUPCON_LR, patience=SUPCON_PATIENCE,
            supcon_weight=SUPCON_WEIGHT, supcon_temperature=SUPCON_TEMPERATURE,
            warmup_fraction=SUPCON_WARMUP_FRACTION, device=DEVICE,
            checkpoint_path=config4_ckpt_path
        )

    results.append(evaluate_on_test(config4_model, test_loader, DEVICE,
                                     "Config 4: + SupCon", class_names))

    print_ablation_table(results)
    pairwise_rows = print_pairwise_significance(results)
    save_ablation_results(results, os.path.join(SAVE_DIR, f"conference_ablation_results_{plm_tag}.csv"))
    save_pairwise_significance(pairwise_rows, os.path.join(SAVE_DIR, f"conference_pairwise_significance_{plm_tag}.csv"))
    per_class_rows_2vs1 = per_class_paired_significance(results[0], results[1], class_names)
    per_class_rows_3vs1 = per_class_paired_significance(results[0], results[2], class_names)
    save_per_class_significance(
        per_class_rows_2vs1 + per_class_rows_3vs1,
        os.path.join(SAVE_DIR, f"per_class_significance_{plm_tag}.csv")
    )


In [ ]:
if __name__ == "__main__":
    run_ablation()